# Part 2 — sweep walkthrough

Runs the base-learner sweep mechanics interactively — one full config end to end, then swaps the base learner and re-runs to expose how the ranking shifts. The full 60-config batch driver lives in `experiments/sweep.py`; this notebook shows the plumbing.

**Reading order.** `00` (this notebook) walks through the mechanics. `01_sweep_results.ipynb` loads the batch output and produces the paper's figures.

## Setup

The `experiments/runner.py` module exposes `run_config`, which takes a `SweepConfig(dataset, meta, base, seed)` and returns a `RunResult` with Qini, bootstrap CI, AUUC, calibration, and the T-column diagnostic. It also writes per-config artifacts under `artifacts/experiments/<name>/`.

First cell brings the machinery into scope.

In [1]:
import sys
from pathlib import Path

# Notebook lives two levels below repo root — expose src/ and experiments/.
REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

from experiments.configs import (
    BASE_LEARNERS, DATASETS, META_LEARNERS, SweepConfig, iter_configs,
)
from experiments.runner import make_learner, run_config, load_arrays

print(f'meta-learners: {META_LEARNERS}')
print(f'base learners: {BASE_LEARNERS}')
print(f'datasets:      {DATASETS}')
print(f'grid size:     {len(META_LEARNERS) * len(BASE_LEARNERS) * len(DATASETS)} configs')

meta-learners: ('propensity', 's', 't', 'cts', 'x', 'cf')
base learners: ('hgb', 'xgb', 'lgbm', 'rf', 'lr')
datasets:      ('criteo', 'hillstrom')
grid size:     60 configs


## One config, end-to-end

Take the paper's motivating case — S-learner with HistGradientBoosting on Criteo — and walk through the pieces. This exact combination is Part 1's degeneracy anecdote. On the 500k train subsample we use in Part 2, HGB no longer degenerates, but the T-column importance and mean|uplift| numbers are still what the paper's figures pivot on.

In [2]:
# Load the Criteo arrays (cached across cells — first call reads the 158MB parquet, subsequent calls hit an in-memory cache).
X_tr, T_tr, Y_tr, X_te, T_te, Y_te = load_arrays('criteo', seed=42)
print(f'train: X {X_tr.shape}  T mean={T_tr.mean():.3f}  Y mean={Y_tr.mean():.4f}')
print(f'test:  X {X_te.shape}  T mean={T_te.mean():.3f}  Y mean={Y_te.mean():.4f}')

train: X (500000, 12)  T mean=0.850  Y mean=0.0472
test:  X (2795919, 12)  T mean=0.850  Y mean=0.0469


/Users/roger.chan/Downloads/uplift-modeling-criteo-main/src/uplift/data.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=random_state))


In [3]:
# Instantiate the meta-learner with a specific base learner.
s_hgb = make_learner(meta='s', base='hgb', seed=42)
print(type(s_hgb).__name__)
print('base estimator:', type(s_hgb.base).__name__)

SLearner
base estimator: HistGradientBoostingClassifier


In [4]:
# Fit, then predict on the held-out test set. Time both.
import time

t0 = time.perf_counter()
s_hgb.fit(X_tr, T_tr, Y_tr)
print(f'fit took {time.perf_counter() - t0:.2f}s')

t0 = time.perf_counter()
uplift = s_hgb.predict_uplift(X_te)
print(f'predict on {len(X_te):,} rows took {time.perf_counter() - t0:.2f}s')
print(f'predicted uplift: mean={uplift.mean():+.5f}  std={uplift.std():.5f}  min={uplift.min():+.4f}  max={uplift.max():+.4f}')

fit took 1.15s


predict on 2,795,919 rows took 1.20s
predicted uplift: mean=+0.00619  std=0.01851  min=-0.3017  max=+0.2639


In [5]:
# Evaluate — Qini + bootstrap CI + calibration.
from uplift.evaluation import bootstrap_qini_ci, qini_curve, qini_coefficient, auuc, uplift_calibration

curve = qini_curve(Y_te, T_te, uplift)
q = qini_coefficient(curve)
ci = bootstrap_qini_ci(Y_te, T_te, uplift, n_boot=200, seed=42)
print(f'Qini coefficient: {q:.2f}  [95% CI: {ci["lo"]:+.1f}, {ci["hi"]:+.1f}]')
print(f'AUUC:             {auuc(curve):.2f}')

Qini coefficient: 8803.67  [95% CI: +8190.7, +9353.8]
AUUC:             21060.73


In [6]:
# The T-column diagnostic — the paper's smoking-gun instrument.
from uplift.diagnostics import s_learner_t_diagnostic

diag = s_learner_t_diagnostic(s_hgb, X_te[:5000])
for k, v in diag.items():
    print(f'{k}: {v}')

t_importance: {'value': 0.03944444444444444, 'kind': 'split_share'}
mean_abs_uplift: 0.006375318688898169
max_abs_uplift: 0.19366965611036974
degenerate: False


**Reading the diagnostic.** `t_importance` here is `split_share` — 3.9% of HGB's internal splits landed on the T column (the last column of the S-learner's augmented feature matrix). `mean_abs_uplift` is 0.006 — the average magnitude of the S-learner's predicted uplift across the test set. If that number were near zero we'd call the S-learner degenerate. It isn't on 500k rows; it *was* on Part 1's 11M rows. Sample-size story.

## Swap the base learner and watch the result change

Same recipe (S-learner), same data. Only the engine changes.

In [7]:
# Rerun the same S-learner recipe with XGBoost, then LightGBM, then LogReg.
import pandas as pd

rows = []
for base_name in ['hgb', 'xgb', 'lgbm', 'rf', 'lr']:
    model = make_learner(meta='s', base=base_name, seed=42)
    t0 = time.perf_counter()
    model.fit(X_tr, T_tr, Y_tr)
    fit_time = time.perf_counter() - t0
    u = model.predict_uplift(X_te)
    q = qini_coefficient(qini_curve(Y_te, T_te, u))
    d = s_learner_t_diagnostic(model, X_te[:5000])
    rows.append({
        'base': base_name,
        'qini': round(q, 1),
        't_importance': round(d['t_importance']['value'], 4) if d['t_importance']['value'] else None,
        't_kind': d['t_importance']['kind'],
        'mean_abs_uplift': round(d['mean_abs_uplift'], 5),
        'fit_time_s': round(fit_time, 2),
    })

pd.DataFrame(rows)

/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:165: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:295: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/l

/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3

/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/roger.chan/Downloads/uplift-modeling-criteo-main/.venv/lib/python3

,base,qini,t_importance,t_kind,mean_abs_uplift,fit_time_s
0,hgb,8803.7,0.0394,split_share,0.00638,1.22
1,xgb,8924.2,0.0082,gain,0.00700,0.45
2,lgbm,8614.2,0.0270,impurity,0.00747,0.98
3,rf,8548.3,0.0023,impurity,0.00364,2.75
4,lr,8104.6,0.0296,coef_abs_share,0.00700,0.14


Same S-learner recipe, five different base learners. The Qini shifts by ~800 points across engines and the T-column attention shifts by an order of magnitude. That's the paper thesis instantiated in a single table.

Note that XGBoost gives T only 0.8% of its native gain but produces the *highest* S-learner Qini (~8924). RF gives T just 0.2% attention but still produces a working uplift ranker (~8548). These two facts together are what makes the T-column importance metric non-trivial — it's an internal signal about how the model built itself, not a proxy for downstream performance.

## The batch driver

The above loop is what `experiments/sweep.py` does — for the full 60-config grid (6 meta × 5 base × 2 datasets), with log-and-skip on failure and resume support via an appended `results.csv`. It's a script, not a notebook, because a live batch of 60 fits taking 15+ minutes doesn't belong in an interactive workflow.

To reproduce the full sweep from scratch (deletes the current results.csv first):

```bash
rm -f artifacts/experiments/results.csv
python -m experiments.sweep                     # both datasets, all 60 configs
python -m experiments.sweep --dataset criteo    # Criteo only, 30 configs
python -m experiments.sweep --dataset hillstrom # Hillstrom only, 30 configs
```

Each config writes to `artifacts/experiments/<name>/` and appends a row to the aggregate `results.csv`. `01_sweep_results.ipynb` loads that CSV and builds the paper's figures.

In [8]:
# The 60-config grid iterator, for reference.
grid = list(iter_configs())
print(f'{len(grid)} configs')
for c in grid[:6]:
    print(f'  {c.name}')
print('  ...')
for c in grid[-3:]:
    print(f'  {c.name}')

60 configs
  criteo_propensity_hgb
  criteo_propensity_xgb
  criteo_propensity_lgbm
  criteo_propensity_rf
  criteo_propensity_lr
  criteo_s_hgb
  ...
  hillstrom_cf_lgbm
  hillstrom_cf_rf
  hillstrom_cf_lr
